
Congrats....again! Based off the amazing work you did on the classification model, you've been promoted to chief of all data scientists in Hollywood, your a total celebrity....among data scientists :).  

Your boss, head of the studio, has now asked you to build a model to predict gross revenue in order to help them decide which movies to invest in.

Once again, you would like to be able to explain the model to mere mortals but need a fairly robust and flexible approach so you've chosen to use decision trees to get started. 

In doing so...you leverage work you've done in the past to get the job done....you're a data scientist after all! 

In [1]:
import pandas as pd
import numpy as np

In [2]:
#1. Load the data
#Sometimes need to set the working directory back out of a folder that we create a file in

#import os
#os.listdir()
#print(os.getcwd())
#os.chdir('c:\\Users\\Brian Wright\\Documents\\3001Python\\DS-3001')

movie_metadata=pd.read_csv("../data/movie_metadata.csv")

movie_metadata.head()



,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0


#2 Ensure all the variables are classified correctly including the target variable and collapse factor variables as needed.

In [7]:
print(movie_metadata.info())

movies = movie_metadata.drop(columns = ['director_name', 'director_facebook_likes', 'actor_3_facebook_likes', 'actor_2_name', 'actor_1_facebook_likes', 'gross', 'actor_1_name', 'movie_title', 'actor_3_name', 'plot_keywords', 'movie_imdb_link', 'actor_2_facebook_likes', 'aspect_ratio'])

movies.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   color                      5024 non-null   object 
 1   director_name              4939 non-null   object 
 2   num_critic_for_reviews     4993 non-null   float64
 3   duration                   5028 non-null   float64
 4   director_facebook_likes    4939 non-null   float64
 5   actor_3_facebook_likes     5020 non-null   float64
 6   actor_2_name               5030 non-null   object 
 7   actor_1_facebook_likes     5036 non-null   float64
 8   gross                      4159 non-null   float64
 9   genres                     5043 non-null   object 
 10  actor_1_name               5036 non-null   object 
 11  movie_title                5043 non-null   object 
 12  num_voted_users            5043 non-null   int64  
 13  cast_total_facebook_likes  5043 non-null   int64

,color,num_critic_for_reviews,duration,genres,num_voted_users,cast_total_facebook_likes,facenumber_in_poster,num_user_for_reviews,language,country,content_rating,budget,title_year,imdb_score,movie_facebook_likes
0,Color,723.0,178.0,Action|Adventure|Fantasy|Sci-Fi,886204,4834,0.0,3054.0,English,USA,PG-13,237000000.0,2009.0,7.9,33000
1,Color,302.0,169.0,Action|Adventure|Fantasy,471220,48350,0.0,1238.0,English,USA,PG-13,300000000.0,2007.0,7.1,0
2,Color,602.0,148.0,Action|Adventure|Thriller,275868,11700,1.0,994.0,English,UK,PG-13,245000000.0,2015.0,6.8,85000
3,Color,813.0,164.0,Action|Thriller,1144337,106759,0.0,2701.0,English,USA,PG-13,250000000.0,2012.0,8.5,164000
4,NaN,NaN,NaN,Documentary,8,143,0.0,NaN,NaN,NaN,NaN,NaN,NaN,7.1,0


#3 Check for missing variables and correct as needed.

In [8]:
movies_1 = movies.dropna() # Dropping null values

movies_1.head()

,color,num_critic_for_reviews,duration,genres,num_voted_users,cast_total_facebook_likes,facenumber_in_poster,num_user_for_reviews,language,country,content_rating,budget,title_year,imdb_score,movie_facebook_likes
0,Color,723.0,178.0,Action|Adventure|Fantasy|Sci-Fi,886204,4834,0.0,3054.0,English,USA,PG-13,237000000.0,2009.0,7.9,33000
1,Color,302.0,169.0,Action|Adventure|Fantasy,471220,48350,0.0,1238.0,English,USA,PG-13,300000000.0,2007.0,7.1,0
2,Color,602.0,148.0,Action|Adventure|Thriller,275868,11700,1.0,994.0,English,UK,PG-13,245000000.0,2015.0,6.8,85000
3,Color,813.0,164.0,Action|Thriller,1144337,106759,0.0,2701.0,English,USA,PG-13,250000000.0,2012.0,8.5,164000
5,Color,462.0,132.0,Action|Adventure|Sci-Fi,212204,1873,1.0,738.0,English,USA,PG-13,263700000.0,2012.0,6.6,24000


#4 Guess what, you don't need to scale the data, because DTs don't require this to be done, they make local greedy decisions...keeps getting easier, go to the next step.

#5 Determine the range and variance of the target variable.

#6 Split your data into test, tune, and train. (80/10/10)

#7 Create the kfold object for cross validation.

#8 Create the scoring metric (several measures) you will use to evaluate your model and the max depth hyperparameter.

#9 Build the regression tree object. 

#10 Use the kfold object and the scoring metric to find the best hyperparameter value for max depth via the grid search method.

#11 Fit the model to the training data.

#12 What is the best depth value?

#13 View the results, comment on how the model performed using several evaluation metrics.

#14 Which variables appear to be contributing the most (variable importance) 

#15 Create a model object using the best model hyperparameter value from the trained regression tree. 

#16 Using the best model predict on the test data and print out the results.

#17 How does the model perform on the test data as compared to the training data?

#18 What five movies are predicted to have the lowest gross revenue from the test set? 

#19 Summarize what you learned along the way and make recommendations on how this could be used moving forward, being careful not to over promise.